# UniChart baseline fine-tuning on ChartQA

Notebook này được cấu trúc để chạy từ file/repo local hiện tại, với kernel đang connect tới GPU của Google Colab. Vì vậy notebook **không clone lại UniChart**; nó dùng trực tiếp `finetune_chartqa.py`, `data/`, và `model/` trong project đang mở.


## 0. Check runtime

Chạy cell này trước để chắc chắn kernel hiện tại đang dùng GPU Colab.


In [9]:
import os, sys, torch
from pathlib import Path

print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")


Python: 3.12.13
Working directory: /content
CUDA available: True
GPU: Tesla T4


## 1. Set local project directory

`PROJECT_DIR` được định nghĩa là thư mục hiện tại của notebook/kernel. Cell này cũng kiểm tra các file UniChart cần thiết có nằm trong thư mục đó không.


In [10]:
from pathlib import Path
import os

PROJECT_DIR = Path.cwd().resolve()
os.chdir(PROJECT_DIR)

print("PROJECT_DIR =", PROJECT_DIR)
print("Files:")
for name in ["finetune_chartqa.py", "data/chartqa_data.py", "model/chartqa_model.py", "requirements.txt"]:
    exists = (PROJECT_DIR / name).exists()
    print(" -", name, exists)

if not (PROJECT_DIR / "finetune_chartqa.py").exists():
    raise FileNotFoundError(
        f"PROJECT_DIR hiện tại không có finetune_chartqa.py: {PROJECT_DIR}. "
        "Hãy cd tới thư mục UniChart local trước khi chạy tiếp."
    )


FileNotFoundError: Không tìm thấy finetune_chartqa.py/data/model trong working directory hiện tại. Hãy set PROJECT_DIR thủ công tới thư mục UniChart local mà kernel có thể truy cập.

## 2. Install dependencies

UniChart gốc khuyến nghị `transformers==4.28.1` và `pytorch-lightning==1.8.5`. Pin thêm `numpy<2` và `torchmetrics==0.11.4` để tránh lỗi tương thích với package mới trên Colab.


In [ ]:
!pip install -q "numpy<2" transformers==4.28.1 pytorch-lightning==1.8.5 torchmetrics==0.11.4 datasets sentencepiece pillow timm nltk


## 3. Configure paths

Ảnh ChartQA sẽ nằm trong `/content/ChartQA` nếu kernel là Colab. Checkpoint fine-tune sẽ lưu ở `/content/output_data` để tránh ghi file lớn vào repo local.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = globals().get("PROJECT_DIR", Path.cwd().resolve())
os.chdir(PROJECT_DIR)

from pathlib import Path

CONTENT_DIR = Path("/content") if Path("/content").exists() else PROJECT_DIR
CHARTQA_DIR = CONTENT_DIR / "ChartQA"
TRAIN_IMAGES = CHARTQA_DIR / "ChartQA Dataset" / "train" / "png"
VAL_IMAGES = CHARTQA_DIR / "ChartQA Dataset" / "val" / "png"
OUTPUT_DIR = CONTENT_DIR / "output_data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("CHARTQA_DIR:", CHARTQA_DIR)
print("TRAIN_IMAGES:", TRAIN_IMAGES)
print("VAL_IMAGES:", VAL_IMAGES)
print("OUTPUT_DIR:", OUTPUT_DIR)


## 4. Download ChartQA images

Labels được load từ Hugging Face dataset `ahmed-masry/chartqa_without_images`; cell này chỉ tải phần ảnh từ repo ChartQA nếu chưa có.


In [ ]:
import subprocess

if not TRAIN_IMAGES.exists() or not VAL_IMAGES.exists():
    if CHARTQA_DIR.exists():
        print("ChartQA directory exists but expected image folders are missing:", CHARTQA_DIR)
        print("Please remove/fix it manually, then rerun this cell.")
    else:
        subprocess.run([
            "git", "clone", "--depth", "1",
            "https://github.com/vis-nlp/ChartQA.git",
            str(CHARTQA_DIR)
        ], check=True)
else:
    print("ChartQA images already exist.")

print("Train images exists:", TRAIN_IMAGES.exists())
print("Val images exists:", VAL_IMAGES.exists())


## 5. Verify data and baseline checkpoint

Cell này kiểm tra Hugging Face dataset, checkpoint `ahmed-masry/unichart-base-960`, và đường dẫn ảnh có khớp nhau không.


In [ ]:
from datasets import load_dataset
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image
import os, torch

BASE_CHECKPOINT = "ahmed-masry/unichart-base-960"
DATASET_NAME = "ahmed-masry/chartqa_without_images"

dataset = load_dataset(DATASET_NAME)
print(dataset)
print("First train sample:", dataset["train"][0])

processor = DonutProcessor.from_pretrained(BASE_CHECKPOINT)
model = VisionEncoderDecoderModel.from_pretrained(BASE_CHECKPOINT)

sample = dataset["train"][0]
img_path = TRAIN_IMAGES / sample["imgname"]
print("Sample image:", img_path)
print("Exists:", img_path.exists())

if not img_path.exists():
    raise FileNotFoundError(img_path)

Image.open(img_path).convert("RGB").resize((480, 360))


## 6. Short fine-tuning run

Chạy 100 steps để kiểm tra toàn bộ pipeline trước. Nếu cell này pass thì mới chạy full fine-tune.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = globals().get("PROJECT_DIR", Path.cwd().resolve())
os.chdir(PROJECT_DIR)

import subprocess, sys

cmd = [
    sys.executable, "finetune_chartqa.py",
    "--data-path", DATASET_NAME,
    "--train-images", str(TRAIN_IMAGES) + "/",
    "--valid-images", str(VAL_IMAGES),
    "--output-dir", str(OUTPUT_DIR),
    "--max-steps", "100",
    "--batch-size", "2",
    "--valid-batch-size", "1",
    "--num-workers", "2",
    "--lr", "5e-5",
    "--check-val-every-n-epoch", "1",
    "--warmup-steps", "20",
    "--checkpoint-steps", "100",
    "--checkpoint-path", BASE_CHECKPOINT,
]

print("Running from:", PROJECT_DIR)
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## 7. Full fine-tuning run

Sau khi smoke test pass, chạy cell này. Với T4 nên bắt đầu `batch-size=2` hoặc `4`; nếu OOM thì giảm batch size.


In [ ]:
from pathlib import Path
import os

PROJECT_DIR = globals().get("PROJECT_DIR", Path.cwd().resolve())
os.chdir(PROJECT_DIR)

import subprocess, sys

cmd = [
    sys.executable, "finetune_chartqa.py",
    "--data-path", DATASET_NAME,
    "--train-images", str(TRAIN_IMAGES) + "/",
    "--valid-images", str(VAL_IMAGES),
    "--output-dir", str(OUTPUT_DIR),
    "--max-steps", "40000",
    "--batch-size", "4",
    "--valid-batch-size", "1",
    "--num-workers", "2",
    "--lr", "5e-5",
    "--check-val-every-n-epoch", "1",
    "--warmup-steps", "100",
    "--checkpoint-steps", "7000",
    "--checkpoint-path", BASE_CHECKPOINT,
]

print("Running from:", PROJECT_DIR)
print(" ".join(cmd))
subprocess.run(cmd, cwd=PROJECT_DIR, check=True)


## 8. Save checkpoints to Google Drive

Chỉ dùng cell này nếu kernel là Colab và bạn muốn giữ checkpoint sau khi runtime tắt.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')

    drive_output = Path('/content/drive/MyDrive/unichart_output_data')
    drive_output.mkdir(parents=True, exist_ok=True)

    subprocess.run(['cp', '-r', str(OUTPUT_DIR) + '/.', str(drive_output)], check=True)
    print('Saved to:', drive_output)
    print(sorted(p.name for p in drive_output.iterdir())[:20])
except ModuleNotFoundError:
    print('google.colab is not available in this runtime. OUTPUT_DIR =', OUTPUT_DIR)


## 9. Inference from latest fine-tuned checkpoint

Cell này dùng checkpoint mới nhất trong `OUTPUT_DIR`; nếu chưa có checkpoint thì fallback về baseline.


In [ ]:
import glob, os, torch
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

saved = sorted(glob.glob(str(OUTPUT_DIR / 'chartqa-checkpoint-*')))
model_name = saved[-1] if saved else BASE_CHECKPOINT
print('Using:', model_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
processor = DonutProcessor.from_pretrained(model_name)
model = VisionEncoderDecoderModel.from_pretrained(model_name).to(device)
model.eval()

def predict_chartqa(image_path, question, max_length=512):
    image = Image.open(image_path).convert('RGB')
    prompt = f'<chartqa> {question} <s_answer>'
    decoder_input_ids = processor.tokenizer(prompt, add_special_tokens=False, return_tensors='pt').input_ids
    pixel_values = processor(image, return_tensors='pt').pixel_values
    with torch.no_grad():
        outputs = model.generate(
            pixel_values.to(device),
            decoder_input_ids=decoder_input_ids.to(device),
            max_length=max_length,
            early_stopping=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            use_cache=True,
            num_beams=4,
            bad_words_ids=[[processor.tokenizer.unk_token_id]],
            return_dict_in_generate=True,
        )
    sequence = processor.batch_decode(outputs.sequences)[0]
    sequence = sequence.replace(processor.tokenizer.eos_token, '').replace(processor.tokenizer.pad_token, '')
    return sequence.split('<s_answer>')[-1].strip()

sample = dataset['val'][0]
image_path = VAL_IMAGES / sample['imgname']
print('Q:', sample['query'])
print('GT:', sample['label'])
print('Pred:', predict_chartqa(image_path, sample['query']))
Image.open(image_path).convert('RGB').resize((480, 360))
